<a href="https://colab.research.google.com/github/BWMIN-Hub/SR_practice/blob/main/notebooks/16_read_properly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reading a satellite image properly

Three questions to answer before the image is used for anything.

In [ ]:
import sys, urllib.request

BASE = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main'
for f in ('lib/intake.py', 'lib/readcheck.py'):
    urllib.request.urlretrieve(f'{BASE}/{f}', f.split('/')[-1])
for m in ('intake', 'readcheck'):
    sys.modules.pop(m, None)
import numpy as np
import rasterio
from readcheck import *

PATH = urllib.request.urlretrieve(f'{BASE}/dataset/intake/input.tif', 'input.tif')[0]

## Q1.  Am I reading it correctly?

Open it, read it, and put bands 1, 2, 3 on screen.  Does a picture come out?

In [ ]:
with rasterio.open(PATH) as d:
    img = d.read()                     # (band, row, col)

print(img.shape, img.dtype)
print('values', img.min(), '-', img.max())

An array arrived, with four bands and values from 1 to 4095.  Now draw it.

In [ ]:
check_read(PATH)

A city.  The file reads and RGB displays, so Q1 is answered.

## Q2.  What type are the numbers?

The array came back as `uint16`.  That is not the same as saying what is in it.

In [ ]:
with rasterio.open(PATH) as d:
    print('dtype', d.dtypes[0])

print('the box holds 0 -', np.iinfo(img.dtype).max)
print('the data uses 0 -', img.max())

In [ ]:
check_type(PATH)

`uint16` is the size of the box, not the amount in it.  The sensor is 12-bit, so the
values stop at 4095 and 94 % of the box is never used.

That is why **the display range has to be chosen**, every time.  Drawn on 0 - 65,535,
as the type suggests, the picture in Q1 would have been black.  It is not a property
of the file.

## Q3.  How big is it?

Two different numbers answer this.

In [ ]:
with rasterio.open(PATH) as d:
    print('pixels    ', d.width, 'x', d.height, ' bands', d.count)
    print('pixel size', d.res)
    print('bounds    ', tuple(round(v, 1) for v in d.bounds), d.crs)

In [ ]:
check_size(PATH)

How much ground it covers decides whether it is the right scene.  How much memory it
takes decides whether it opens at all — on disk it is 21 MB, in memory 46 MB, because
the file is compressed and the array is not.